# SVM Classification Practical — Simple Exam Notebook

**Scenario:** Breast Cancer Detection  
**Dataset:** `breast_cancer.csv`  
**Target:** `target` (`0` = Malignant, `1` = Benign)

## What we will do
1. Load and understand the dataset.
2. Explore missing values, target balance, correlations and outliers.
3. Separate predictors (`X`) and target (`y`).
4. Build SVM without PCA and tune it using 5-fold Stratified cross-validation.
5. Build SVM with PCA and tune it in the same way.
6. Compare both models and run a paired t-test (`scipy.stats.ttest_rel`).

**Important:** Keep `breast_cancer.csv` in the same folder as this notebook. All scaling and PCA are placed inside pipelines so cross-validation evaluates them cleanly.


## 1. Import the tools

These are the exact tools from the cheatsheet: pandas, numpy, matplotlib, seaborn, StandardScaler, Pipeline, SVC, PCA, GridSearchCV, StratifiedKFold, classification metrics and ttest_rel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ttest_rel


## 2. Load the dataset and inspect it

- `head()` shows the first rows.
- `shape` shows rows and columns.
- `info()` shows data types and non-missing counts.
- `isnull().sum()` counts missing values.
- `describe()` gives summary statistics.


In [ ]:
df = pd.read_csv("breast_cancer.csv")

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nNumerical summary:")
display(df.describe())


## 3. Explore the target and features

Check whether classes are balanced using `value_counts()`, plot target distribution, and display a correlation heatmap.


In [ ]:
target_col = "target"

print("Class counts:")
print(df[target_col].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Target Distribution (0 = Malignant, 1 = Benign)")
plt.show()

plt.figure(figsize=(10, 8))
corr = df.iloc[:, :10].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (First 10 Features)")
plt.show()


## 4. Separate features (X) and target (y)


In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


## 5. Build SVM without PCA and tune using GridSearchCV

We scale features using `StandardScaler` and classify using `SVC` with an RBF kernel. Grid search tunes `C` and `gamma` across 5 Stratified folds.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_no_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=42))
])

param_grid_no_pca = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid_no_pca = GridSearchCV(
    estimator=pipe_no_pca,
    param_grid=param_grid_no_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_no_pca.fit(X, y)

print("Best Parameters (No PCA):", grid_no_pca.best_params_)
print("Best CV Accuracy (No PCA):", round(grid_no_pca.best_score_, 4))


## 6. Build SVM with PCA and tune using GridSearchCV

Here we add `PCA(n_components=0.95)` to retain 95% of the total variance while reducing dimensionality.


In [ ]:
pipe_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("svm", SVC(kernel="rbf", random_state=42))
])

param_grid_pca = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto"]
}

grid_pca = GridSearchCV(
    estimator=pipe_pca,
    param_grid=param_grid_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_pca.fit(X, y)

print("Best Parameters (With PCA):", grid_pca.best_params_)
print("Best CV Accuracy (With PCA):", round(grid_pca.best_score_, 4))


## 7. Compare Models & Statistical Significance Test (Paired t-test)

We compare the 5-fold cross-validation scores using `ttest_rel` from `scipy.stats` as given in the cheatsheet.


In [ ]:
best_no_pca = grid_no_pca.best_estimator_
best_pca = grid_pca.best_estimator_

scores_no_pca = cross_val_score(best_no_pca, X, y, cv=cv, scoring="accuracy")
scores_pca = cross_val_score(best_pca, X, y, cv=cv, scoring="accuracy")

print("Fold Scores (No PCA):", np.round(scores_no_pca, 4))
print("Mean Accuracy (No PCA):", round(scores_no_pca.mean(), 4))
print("\nFold Scores (With PCA):", np.round(scores_pca, 4))
print("Mean Accuracy (With PCA):", round(scores_pca.mean(), 4))

t_stat, p_val = ttest_rel(scores_no_pca, scores_pca)
print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Significant difference between models (p < 0.05)")
else:
    print("No significant difference between models (p >= 0.05)")


## 8. Final Evaluation & Confusion Matrix


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

best_no_pca.fit(X_train, y_train)
y_pred = best_no_pca.predict(X_test)

print("Classification Report (Best Model):")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
